<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L56_Continual_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L56: Continual Learning - Updating Models Without Forgetting

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 56 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Understand catastrophic forgetting when fine-tuning on new data
- Apply regularization (e.g. EWC, replay) or small learning rates to retain old knowledge
- Implement a simple replay buffer for continual fine-tuning

---

## Concept: Continual Learning

**Continual learning**: train on new tasks/data over time without forgetting previous ones. **Catastrophic forgetting**: new training overwrites old weights. **Mitigations**: low LR, elastic weight consolidation (EWC), replay (mix old and new data), or parameter isolation (e.g. adapters per task). We illustrate replay.

---

In [ ]:
!pip install transformers torch datasets -q
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Replay Buffer (Old Data)

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

old_data = load_dataset("imdb", split="train", trust_remote_code=True).select(range(200))
old_data = old_data.map(tokenize_fn, batched=True, remove_columns=["text"])
old_data.set_format("torch")

new_data = load_dataset("ag_news", split="train", trust_remote_code=True).select(range(200))
new_data = new_data.map(tokenize_fn, batched=True, remove_columns=["text"])
new_data.set_format("torch")

replay_ratio = 0.3
replay_size = int(len(new_data) * replay_ratio / (1 - replay_ratio))
from torch.utils.data import ConcatDataset, Subset
replay_subset = Subset(old_data, range(min(replay_size, len(old_data))))
continual_ds = ConcatDataset([new_data, replay_subset])
print("Continual dataset: new task + replay of old task.")

## Part 2: Fine-Tune with Replay

---

In [ ]:
# Note: old_data is 2-class (imdb), new_data is 4-class (ag_news). For same task, use same num_labels.
# Here we just show mixing datasets; in practice keep same label space or use multi-head.
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)
args = TrainingArguments(output_dir="./out_continual_l56", num_train_epochs=1, per_device_train_batch_size=8, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=new_data)
trainer.train()
print("With full replay: train_dataset=continual_ds. Replay helps retain old task performance.")

## Part 3: EWC (Concept)

---

In [ ]:
print("EWC: after task A, compute Fisher diagonal; when training on B, add penalty for changing important params for A.")

## Exercises

1. Train on task A, then on B with 0% vs 30% replay; measure accuracy on A and B.
2. Implement a simple EWC: store diagonal Fisher, add L2 penalty to loss.
3. Use LoRA for task B and freeze base; compare forgetting vs full fine-tune.

---

## Key Takeaways

1. Replay (mix old + new data) is a simple way to reduce forgetting.
2. EWC and similar regularize important weights for previous tasks.
3. Adapters/LoRA per task isolate parameters and avoid overwriting.

---

## Next Lesson

**L57: Model Merging** – Weight averaging and merge strategies.

---